# 02 — Descarga, carga y revisión de calidad de los datos

**Objetivo:** traer el dataset Prosper desde Kaggle, cargarlo sin modificarlo, documentar sus variables y medir su calidad.

**Resultados que guarda:**
- `data/interim/prosper_crudo.parquet`: datos con fechas ya tipadas (los usa el notebook 03)
- `docs/diccionario_variables.csv`: diccionario técnico de las columnas
- `docs/calidad_resumen.csv`: indicadores de calidad
- `docs/figuras/02_nulos_top25.png`: gráfica de valores faltantes

## 1. Preparar rutas e importaciones

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import RUTAS, crear_carpetas
from src.datos import (descargar_prosper, buscar_csv_crudo, cargar_crudo,
                       perfil_columnas, resumen_calidad, COLUMNAS_FUGA)

crear_carpetas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 120)

## 2. Descargar el dataset (solo la primera vez)
Si el CSV ya está en `data/raw/`, no se vuelve a descargar. Si falla la descarga automática, usa la opción manual de `data/README.md` y vuelve a ejecutar.

In [ ]:
if list(RUTAS['data_raw'].glob('*.csv')):
    print('El CSV ya existe en data/raw/, no se descarga de nuevo.')
else:
    archivos = descargar_prosper()
    print('Descargado:', [a.name for a in archivos])

ruta_csv = buscar_csv_crudo()
print('Archivo a usar:', ruta_csv.name, f'({ruta_csv.stat().st_size / 1024**2:.1f} MB)')

## 3. Cargar y verificar dimensiones
El enunciado habla de ~114.000 filas y 81 columnas: aquí lo **verificamos** en lugar de suponerlo.

In [ ]:
df = cargar_crudo(ruta_csv)
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
df.head(3).T  # transpuesto: una fila por columna, más fácil de leer con 81 columnas

## 4. Tipos de datos (`df.info()`)
📋 **Pega esta salida en el chat**: con ella se arma el diccionario de negocio y se decide el rol de cada variable.

In [ ]:
df.info(verbose=True, show_counts=True)

## 5. Resumen de calidad

In [ ]:
calidad = resumen_calidad(df)
calidad

## 6. Valores faltantes por columna
Las columnas con muchos nulos no se descartan automáticamente: en Prosper, varias variables solo existen para créditos de cierto periodo, y el nulo puede tener significado.

In [ ]:
nulos = (df.isna().mean() * 100).sort_values(ascending=False)
print('Columnas con al menos un nulo:', (nulos > 0).sum())

top = nulos.head(25).sort_values()
fig, ax = plt.subplots(figsize=(8, 7))
top.plot.barh(ax=ax, color='#4C72B0')
ax.set_xlabel('% de valores faltantes')
ax.set_title('Top 25 columnas con más valores faltantes')
plt.tight_layout()
fig.savefig(RUTAS['figuras'] / '02_nulos_top25.png', dpi=150)
plt.show()

nulos[nulos > 0].round(2).to_frame('pct_nulos')

## 7. Estado del crédito (`LoanStatus`)
De aquí sale la **población objetivo** del proyecto (Chargedoff y Defaulted).

In [ ]:
estado = df['LoanStatus'].value_counts(dropna=False).to_frame('creditos')
estado['pct'] = (estado['creditos'] / len(df) * 100).round(2)
estado

## 8. Rango de fechas
Necesario para la **validación temporal** (entrenar con créditos antiguos y validar con recientes).

In [ ]:
cols_fecha = df.select_dtypes(include='datetime').columns
pd.DataFrame({
    'minima': df[cols_fecha].min(),
    'maxima': df[cols_fecha].max(),
    'pct_nulos': (df[cols_fecha].isna().mean() * 100).round(2),
})

## 9. Verificar las columnas de fuga de datos
Confirmamos que existen con esos nombres exactos. Se usarán **solo** para construir la variable objetivo y el ROI.

In [ ]:
for col in COLUMNAS_FUGA:
    print(f"{'✅' if col in df.columns else '❌ NO EXISTE'}  {col}")

## 10. Guardar resultados
El notebook 03 leerá el parquet: es más rápido que el CSV y conserva los tipos de fecha.

In [ ]:
diccionario = perfil_columnas(df)
diccionario.to_csv(RUTAS['docs'] / 'diccionario_variables.csv', index=False, encoding='utf-8-sig')
calidad.to_csv(RUTAS['docs'] / 'calidad_resumen.csv', index=False, encoding='utf-8-sig')

salida = RUTAS['data_interim'] / 'prosper_crudo.parquet'
df.to_parquet(salida, index=False)

print('✅ Guardado:')
print('  ', salida.relative_to(RUTAS['raiz']), f'({salida.stat().st_size / 1024**2:.1f} MB)')
print('   docs/diccionario_variables.csv')
print('   docs/calidad_resumen.csv')
print('   docs/figuras/02_nulos_top25.png')